# 🏥 Hospital Readmission Risk Analyzer
### Identifying high-risk diabetic patients before they are discharged

**Author:** Utsav Raj  
**Program:** PGDM-BM | Aspiring Business Analyst  
**Dataset:** Diabetes 130-US Hospitals (1999–2008) — UCI Machine Learning Repository  
**Records:** 101,766 patient encounters across 130 US hospitals  

---

## 🎯 Business Problem
Hospital readmissions within 30 days are a major financial and operational challenge.
Under the **Hospital Readmissions Reduction Program (HRRP)**, US Medicare penalizes
hospitals financially for excessive 30-day readmission rates.

**The question:** Can we identify HIGH RISK patients BEFORE discharge so hospitals
can intervene early and prevent costly readmissions?

## 📋 Research Hypotheses
Before touching the data, we formed 4 hypotheses to test:
- H1 → Older patients have higher readmission rates
- H2 → More diagnoses = higher readmission risk
- H3 → More medications = higher readmission risk
- H4 → Previous inpatient visits predict future readmission

In [41]:
# Cell 1: Install & Import Libraries
!pip install plotly --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print('=' * 55)
print('   Hospital Readmission Risk Analyzer')
print('   Identifying high-risk patients before discharge')
print('=' * 55)

   Hospital Readmission Risk Analyzer
   Identifying high-risk patients before discharge


In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
# Cell 2: Load Dataset
diabetic = '/content/drive/MyDrive/DataSet for practice/diabetic_data.csv'
df = pd.read_csv(diabetic)

print(f'Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head()

Dataset loaded: 101,766 rows, 50 columns


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Phase 1 — Data Cleaning
Raw hospital data is messy. Missing values are stored as `?` instead of proper NaN.
We need to clean this before any analysis.

In [44]:
# Cell 3: Replace ? with NaN
df = df.replace('?', np.nan)

print('Missing values after replacement:')
missing = df.isnull().sum()
print(missing[missing > 0])

Missing values after replacement:
race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
max_glu_serum        96420
A1Cresult            84748
dtype: int64


In [45]:
# Cell 4: Drop High-Missing Columns & Clean Rows
# Drop columns with more than 40% missing values
df.drop(columns=['weight', 'payer_code', 'medical_specialty',
                 'max_glu_serum', 'A1Cresult'], inplace=True)

# Drop remaining rows with any missing values
df.dropna(inplace=True)



In [46]:
# Cell 5: Create Binary Readmission Target
# <30 days = HIGH RISK (1) | everything else = NOT high risk (0)
df['readmit_binary'] = np.where(df['readmitted'] == '<30', 1, 0)

print('Readmission breakdown:')
print(df['readmit_binary'].value_counts())
print()
print('As percentage:')
print(df['readmit_binary'].value_counts(normalize=True) * 100)

Readmission breakdown:
readmit_binary
0    86987
1    11066
Name: count, dtype: int64

As percentage:
readmit_binary
0    88.714267
1    11.285733
Name: proportion, dtype: float64


### 💡 Business Insight 1 — Readmission Rate
**11.28% of all diabetic patients are readmitted within 30 days.**

In a hospital treating 100 patients per day, approximately **11 patients per day**
trigger a Medicare financial penalty.

Early identification of these patients could save the hospital significant
financial penalties and improve patient outcomes.

## Phase 2 — Hypothesis Testing
Now we test each of our 4 hypotheses against real data.

In [47]:
# Cell 6: H1 — Age vs Readmission
age_risk = df.groupby('age')['readmit_binary'].mean().sort_values(ascending=False)
print('Readmission rate by age group:')
print(age_risk)

Readmission rate by age group:
age
[20-30)     0.144114
[80-90)     0.120465
[70-80)     0.118707
[30-40)     0.114994
[60-70)     0.112797
[90-100)    0.110048
[40-50)     0.107933
[50-60)     0.098101
[10-20)     0.066524
[0-10)      0.015385
Name: readmit_binary, dtype: float64


### 💡 Business Insight 2 — Age vs Readmission
**Surprising finding:** Young adults (20-30) have the **HIGHEST** readmission rate
at 14.4% — higher than elderly patients.

This **contradicts** the common assumption that older = higher risk.

**Likely cause:** Medication non-compliance and lifestyle factors in young adults —
they tend to underestimate the seriousness of their condition.

**Recommendation:** Hospitals should provide stronger discharge counseling
and follow-up calls specifically for the 20-30 age group.

In [48]:
# Cell 7: H2 — Number of Diagnoses vs Readmission
diag_risk = df.groupby('number_diagnoses')['readmit_binary'].mean().sort_values(ascending=False)
print('Readmission rate by number of diagnoses:')
print(diag_risk)

Readmission rate by number of diagnoses:
number_diagnoses
11    0.272727
10    0.187500
13    0.187500
14    0.142857
15    0.125000
9     0.124469
8     0.117828
12    0.111111
7     0.108164
6     0.105026
16    0.100000
5     0.091579
4     0.082820
3     0.073428
Name: readmit_binary, dtype: float64


### 💡 Business Insight 3 — Number of Diagnoses vs Readmission
**Clear pattern confirmed:** More diagnoses = higher readmission risk.

- Patients with **9+ diagnoses** have >12% readmission rate
- Patients with **3 diagnoses** have only 7.3% readmission rate

**Recommendation:** Flag patients with **7+ diagnoses** for extended hospital stay
or mandatory follow-up appointment before discharge.

In [49]:
# Cell 8: H3 — Medications vs Readmission
med_risk = df.groupby('num_medications')['readmit_binary'].mean()
print('Readmission rate for patients on 20+ medications:')
print(med_risk[med_risk.index > 20])

Readmission rate for patients on 20+ medications:
num_medications
21    0.129247
22    0.135290
23    0.128842
24    0.139185
25    0.119589
26    0.129217
27    0.123317
28    0.120332
29    0.144778
30    0.119710
31    0.131994
32    0.110561
33    0.099190
34    0.142534
35    0.167102
36    0.128114
37    0.157895
38    0.135747
39    0.107317
40    0.137931
41    0.115942
42    0.120968
43    0.134921
44    0.071429
45    0.125000
46    0.146067
47    0.166667
48    0.107143
49    0.114754
50    0.076923
51    0.046512
52    0.075472
53    0.081081
54    0.030303
55    0.193548
56    0.081081
57    0.076923
58    0.166667
59    0.150000
60    0.150000
61    0.142857
62    0.066667
63    0.076923
64    0.142857
65    0.000000
66    0.000000
67    0.285714
68    0.285714
69    0.000000
70    0.500000
72    1.000000
74    0.000000
75    0.000000
79    0.000000
81    1.000000
Name: readmit_binary, dtype: float64


### 💡 Business Insight 4 — Medications vs Readmission
Patients on **20-40 medications** show consistently elevated readmission rates of **11-16%**.

**Important caveat:** Extreme values (72, 81 medications) show 100% readmission
but represent only 1-2 patients each — statistically unreliable (small sample size problem).

**Recommendation:** Flag patients on **20+ medications** for pharmacy review before
discharge to check for drug interactions that could cause readmission.

In [50]:
# Cell 9: H4 — Previous Admissions vs Readmission
inpatient_risk = df.groupby('number_inpatient')['readmit_binary'].mean()
print('Readmission rate by number of previous inpatient visits:')
print(inpatient_risk)

Readmission rate by number of previous inpatient visits:
number_inpatient
0     0.085450
1     0.129805
2     0.173427
3     0.203825
4     0.236694
5     0.312968
6     0.345992
7     0.357143
8     0.448980
9     0.423423
10    0.416667
11    0.666667
12    0.484848
13    0.500000
14    0.400000
15    1.000000
16    0.200000
18    0.000000
19    0.500000
21    1.000000
Name: readmit_binary, dtype: float64


### 💡 Business Insight 5 — Previous Admissions vs Readmission
**STRONGEST PREDICTOR FOUND:**

- Patients with **0 previous admissions** → only 8.5% readmission rate
- Patients with **3+ previous admissions** → crosses 20% threshold
- Patients with **8+ previous admissions** → nearly 1 in 2 readmitted!

This is a **5x increase** from baseline to high-history patients.

**Recommendation:** Any patient with **3+ previous inpatient visits** should be
automatically flagged as HIGH RISK and assigned a dedicated care coordinator.

## Phase 3 — Risk Scoring System
Using our 4 confirmed risk factors, we build a clinical risk score
to classify every patient as LOW, MEDIUM, or HIGH risk.

In [51]:
# Cell 10: Risk Score Calculator
df['risk_score'] = 0

# Rule 1 — Previous admissions (strongest predictor)
df.loc[df['number_inpatient'] >= 3, 'risk_score'] += 3
df.loc[df['number_inpatient'] >= 6, 'risk_score'] += 2

# Rule 2 — Number of diagnoses
df.loc[df['number_diagnoses'] >= 7, 'risk_score'] += 2

# Rule 3 — Number of medications
df.loc[df['num_medications'] >= 20, 'risk_score'] += 1

# Rule 4 — Young adult (highest surprise risk group)
df.loc[df['age'] == '[20-30)', 'risk_score'] += 1

def classify_risk(score):
    if score >= 6:
        return 'HIGH'
    elif score >= 3:
        return 'MEDIUM'
    else:
        return 'LOW'

df['risk_level'] = df['risk_score'].apply(classify_risk)

print('Patient count by risk level:')
print(df['risk_level'].value_counts())
print()
print('Actual readmission rate by risk level:')
print((df.groupby('risk_level')['readmit_binary'].mean() * 100).round(2))

Patient count by risk level:
risk_level
LOW       70812
MEDIUM    24415
HIGH       2826
Name: count, dtype: int64

Actual readmission rate by risk level:
risk_level
HIGH      29.19
LOW        9.64
MEDIUM    13.99
Name: readmit_binary, dtype: float64


### 🏆 KEY PROJECT FINDING — Risk Scoring System Performance

| Risk Level | Patients | Readmission Rate | Meaning |
|---|---|---|---|
| LOW | 70,812 | 9.6% | 9 in 100 readmitted |
| MEDIUM | 24,415 | 14.0% | 14 in 100 readmitted |
| HIGH | 2,826 | 29.2% | 29 in 100 readmitted |

**HIGH risk patients are 3x more likely to be readmitted than LOW risk patients.**

**Business Impact:**
Hospital can focus intensive follow-up resources on just **2,826 HIGH risk patients (2.9% of total)**
instead of all 98,053 patients — a targeted approach that could prevent hundreds of
Medicare penalty-triggering readmissions annually.

## Phase 4 — Visualizations

In [52]:
# Cell 11: Chart 1 — Patient Risk Distribution
risk_counts = df['risk_level'].value_counts().reset_index()
risk_counts.columns = ['Risk Level', 'Count']

fig = px.bar(
    risk_counts,
    x='Risk Level',
    y='Count',
    color='Risk Level',
    color_discrete_map={
        'LOW'   : '#4CAF50',
        'MEDIUM': '#FF9800',
        'HIGH'  : '#F44336'
    },
    title='Patient Risk Distribution — Hospital Readmission Risk Analyzer'
)
fig.show()

In [53]:
# Cell 12: Chart 2 — Readmission Rate by Risk Level
risk_rates = df.groupby('risk_level')['readmit_binary'].mean().reset_index()
risk_rates.columns = ['Risk Level', 'Readmission Rate']
risk_rates['Readmission Rate'] = (risk_rates['Readmission Rate'] * 100).round(2)

fig2 = px.bar(
    risk_rates,
    x='Risk Level',
    y='Readmission Rate',
    color='Risk Level',
    color_discrete_map={
        'LOW'   : '#4CAF50',
        'MEDIUM': '#FF9800',
        'HIGH'  : '#F44336'
    },
    title='Readmission Rate by Risk Level — HIGH risk patients are 3x more likely to return',
    text='Readmission Rate'
)
fig2.update_traces(texttemplate='%{text}%', textposition='outside')
fig2.update_layout(yaxis_title='Readmission Rate (%)')
fig2.show()

In [54]:
# Cell 13: Chart 3 — Previous Admissions Trend
inpatient_risk = df.groupby('number_inpatient')['readmit_binary']\
                   .mean().reset_index()
inpatient_risk.columns = ['Previous Admissions', 'Readmission Rate']
inpatient_risk['Readmission Rate'] = (inpatient_risk['Readmission Rate'] * 100).round(2)
inpatient_risk = inpatient_risk[inpatient_risk['Previous Admissions'] <= 10]

fig3 = px.line(
    inpatient_risk,
    x='Previous Admissions',
    y='Readmission Rate',
    markers=True,
    title='More Previous Admissions = Higher Readmission Risk (Strongest Predictor Found)'
)
fig3.update_traces(line_color='#F44336', line_width=2.5)
fig3.update_layout(yaxis_title='Readmission Rate (%)')
fig3.show()

---
## 📊 Final Summary — Key Business Insights & Recommendations

| # | Finding | Recommendation |
|---|---|---|
| 1 | 11.28% of diabetic patients readmit within 30 days | Implement pre-discharge risk screening for all patients |
| 2 | Young adults (20-30) have HIGHEST readmission rate at 14.4% | Stronger discharge counseling for 20-30 age group |
| 3 | Patients with 9+ diagnoses have >12% readmission rate | Flag 7+ diagnoses patients for extended stay |
| 4 | 20+ medications = elevated readmission risk | Pharmacy review before discharge for complex medication regimens |
| 5 | 3+ previous admissions crosses 20% readmission threshold | Auto-flag repeat patients as HIGH RISK |
| 6 | HIGH risk patients are 3x more likely to readmit than LOW risk | Focus intensive follow-up on 2,826 HIGH risk patients only |

---

## 🎯 Conclusion
This analysis successfully built a **rule-based risk scoring system** that identifies
HIGH risk diabetic patients before discharge — without requiring any machine learning.

By focusing intensive follow-up resources on just **2.9% of patients** (HIGH risk group),
hospitals can significantly reduce 30-day readmission rates and avoid Medicare financial penalties.

**Previous inpatient admission history** emerged as the single strongest predictor,
confirming the principle: past behaviour predicts future behaviour, even in healthcare.

---
*Analysis by Utsav Raj | PGDM-BM | Aspiring Business Analyst*  
*Tools: Python | pandas | numpy | plotly | Google Colab*  
*Dataset: UCI Machine Learning Repository — Diabetes 130-US Hospitals*